In [5]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

# 0.Preprocessing

In [6]:
# ========= 路径（你自己改） =========
roi_path  = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/processed/06.active_map/res/roi_map/working-memory_roi_left_top15_kwx.csv"      # 你截图里的 ROI 文件
expr_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_activate_kwx.csv"   # 原始 expr
out_path  = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_top15_activate_kwx.csv"

# ========= 1) 读 ROI 文件 =========
roi_df = pd.read_csv(roi_path, header=None)   # 如果你没有表头
# 如果你的 ROI 文件有表头，用下面这一行替换：
# roi_df = pd.read_csv(roi_path)

roi_names = (roi_df.iloc[:, 0].astype(str).str.strip().unique())

print("ROI file contains", len(roi_names), "ROIs")

# ========= 2) 读 expr 文件 =========
expr = pd.read_csv(expr_path, index_col=0)
expr.index = expr.index.astype(str).str.strip()

print("Original expr shape:", expr.shape)

# ========= 3) 按 ROI 名筛选 expr =========
common_rois = expr.index.intersection(roi_names)
print("Matched ROIs in expr:", len(common_rois))

expr_filtered = expr.loc[common_rois]

print("Filtered expr shape:", expr_filtered.shape)

# ========= 4) 保存新文件 =========
expr_filtered.to_csv(out_path)

print("Saved filtered expr to:")
print(out_path)

ROI file contains 10 ROIs
Original expr shape: (64, 1563)
Matched ROIs in expr: 10
Filtered expr shape: (10, 1563)
Saved filtered expr to:
/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_top15_activate_kwx.csv


In [7]:
# 1. 读取 ROI × Gene 表达矩阵
expr = pd.read_csv("/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_top15_activate_kwx.csv",index_col=0)
# expr: 10 × G
print(expr.shape)

# 2. 对每个基因在 ROI 维度上做 z-score
# axis=0 表示对每一列（gene）标准化
expr_z = pd.DataFrame(zscore(expr, axis=0, nan_policy="omit"),index=expr.index,columns=expr.columns)

(10, 1563)


# 1.Co-exprssion

In [8]:
# 3. 计算 ROI × ROI 的基因共表达矩阵（Pearson）
# 每一行是一个 ROI 的“基因表达模式”
gene_coexpression = expr_z.T.corr(method="pearson")

# 2.Save

In [10]:
# 4. 保存结果
gene_coexpression.to_csv("/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/02.co_expression/gene_co_expr_matrix_top10_kwx.csv")

print("Gene co-expression matrix saved.")
print(gene_coexpression.shape)

Gene co-expression matrix saved.
(10, 10)


# 3.后9个选择区间的基因共表达

In [1]:
import os
import pandas as pd
import numpy as np
from scipy.stats import zscore

# =========================================
# 1. 路径设置
# =========================================
base_dir = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res"
gene_sel_dir = os.path.join(base_dir, "01.gene_selection")
coexp_dir = os.path.join(base_dir, "02.co_expression")
os.makedirs(coexp_dir, exist_ok=True)

roi_path = "/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/processed/06.active_map/res/roi_map/working-memory_roi_left_top15_kwx.csv"

# 你要处理的 9 个区间
percent_ranges = [
    (10, 20),
    (20, 30),
    (30, 40),
    (40, 50),
    (50, 60),
    (60, 70),
    (70, 80),
    (80, 90),
    (90, 100),
]

# =========================================
# 2. 读取 ROI 文件
#    保持和你原代码一致：读取第一列作为 ROI 名称
# =========================================
roi_df = pd.read_csv(roi_path, header=None)
roi_names = roi_df.iloc[:, 0].astype(str).str.strip().unique()

print("ROI file contains", len(roi_names), "ROIs")

# =========================================
# 3. 循环处理每个 expr 文件
# =========================================
for start_p, end_p in percent_ranges:
    # ---------- 输入输出文件名 ----------
    expr_in_path = os.path.join(
        gene_sel_dir,
        f"expr_left_top{start_p}_{end_p}_activate_kwx.csv"
    )

    expr_top15_out_path = os.path.join(
        gene_sel_dir,
        f"expr_left_top{start_p}_{end_p}_top15_activate_kwx.csv"
    )

    coexp_out_path = os.path.join(
        coexp_dir,
        f"gene_co_expr_matrix_top{start_p}_{end_p}_kwx.csv"
    )

    print("\n=====================================")
    print(f"Processing {start_p}-{end_p}% genes")
    print("Input expr:", expr_in_path)

    # =========================================
    # Step 1: 读取 expr 文件
    # =========================================
    expr = pd.read_csv(expr_in_path, index_col=0)
    expr.index = expr.index.astype(str).str.strip()

    print("Original expr shape:", expr.shape)

    # =========================================
    # Step 2: 按 ROI 名筛选 expr
    # 和你原逻辑完全一致
    # =========================================
    common_rois = expr.index.intersection(roi_names)
    print("Matched ROIs in expr:", len(common_rois))

    expr_filtered = expr.loc[common_rois]
    print("Filtered expr shape:", expr_filtered.shape)

    # 保存 top15 ROI 版本的 expr
    expr_filtered.to_csv(expr_top15_out_path)
    print("Saved filtered expr to:")
    print(expr_top15_out_path)

    # =========================================
    # Step 3: 对每个基因在 ROI 维度上做 z-score
    # 和你原逻辑完全一致
    # =========================================
    expr_z = pd.DataFrame(
        zscore(expr_filtered, axis=0, nan_policy="omit"),
        index=expr_filtered.index,
        columns=expr_filtered.columns
    )

    # =========================================
    # Step 4: 计算 ROI × ROI 的基因共表达矩阵（Pearson）
    # 和你原逻辑完全一致
    # 每一行是一个 ROI 的基因表达模式
    # =========================================
    gene_coexpression = expr_z.T.corr(method="pearson")

    # =========================================
    # Step 5: 保存结果
    # =========================================
    gene_coexpression.to_csv(coexp_out_path)

    print("Gene co-expression matrix saved:")
    print(coexp_out_path)
    print("Co-expression matrix shape:", gene_coexpression.shape)

print("\nAll done.")

ROI file contains 10 ROIs

Processing 10-20% genes
Input expr: /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_20_activate_kwx.csv
Original expr shape: (64, 1563)
Matched ROIs in expr: 10
Filtered expr shape: (10, 1563)
Saved filtered expr to:
/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top10_20_top15_activate_kwx.csv
Gene co-expression matrix saved:
/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/02.co_expression/gene_co_expr_matrix_top10_20_kwx.csv
Co-expression matrix shape: (10, 10)

Processing 20-30% genes
Input expr: /Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene_selection/expr_left_top20_30_activate_kwx.csv
Original expr shape: (64, 1564)
Matched ROIs in expr: 10
Filtered expr shape: (10, 1564)
Saved filtered expr to:
/Volumes/KWX/Grad/Programs/2025/202505/20250529/activity_gene_analysis/res/01.gene